# D1-11 Optional - Sea-floor transformation flows and custom LCIA method

This notebook shows how to add new sea-floor transformation elementary flows to a separate biosphere database, characterize those flows with a new LCIA method, and use them in a small foreground process.

Audience:
- Participants who already know the Brightway concepts of projects, databases, activities, exchanges, and methods.

Prerequisites:
- The `bw` conda environment is active.
- The `aalborg-rlcia-2026` Brightway project exists, or you are comfortable letting Brightway create it.

Learning goals:
- Create an additional biosphere database for sea-floor transformation flows that do not exist in the existing biosphere database.
- Register a custom LCIA method with characterization factors for those flows.
- Create a fake process database that emits/uses the new flows.
- Run an LCA and check the score against a manual calculation.


## Workflow

1. Select the course Brightway project.
2. Remove only the scratch objects used by this notebook, so the notebook is rerunnable.
3. Create two elementary flows in a new biosphere database.
4. Register a custom LCIA method that gives both flows a characterization factor.
5. Create one fake foreground process with biosphere exchanges to those flows.
6. Run LCI and LCIA, then compare Brightway's score with a manual calculation.


## 1. Imports and names

The database and method names below are deliberately specific to this notebook. The cleanup cell removes only these names, and does not touch the regular course databases.


In [1]:
import math

import bw2calc as bc
import bw2data as bd


PROJECT_NAME = "aalborg-rlcia-2026"

# Scratch database names used only in this notebook.
EXTRA_BIOSPHERE_DB = "extra-biosphere"
PROCESS_DB = "some-db"

# Brightway method names are tuples. Use a tuple that is unlikely to collide
# with installed LCIA methods.
CUSTOM_METHOD = (
    "additional LCIA method",
    "marine seafloor transformation midpoint",
)


bd.projects.set_current(PROJECT_NAME)

print("Current Brightway project:", bd.projects.current)
print("Existing databases:", sorted(bd.databases))

/opt/homebrew/Caskroom/miniforge/base/envs/bw/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


Current Brightway project: aalborg-rlcia-2026
Existing databases: ['bafu', 'ecoinvent-3.10-biosphere', 'extra-biosphere', 'h2_electrolysis', 'some-db']


## 2. Create an additional biosphere database

Brightway does not require a special biosphere class. A biosphere database is a normal database whose datasets represent elementary flows.

The two flows below are intentionally artificial. They follow the metadata pattern used by ecoinvent transformation flows:
- unit: `square meter`
- type: `natural resource`
- categories: `("natural resource", "land")`


In [2]:
extra_biosphere_data = {
    (EXTRA_BIOSPHERE_DB, "sea-floor-transformation-coastal-area"): {
        "name": "Sea floor transformation, coastal area",
        "unit": "square meter",
        "categories": ("natural resource", "land"),
        "type": "natural resource",
        "code": "sea-floor-transformation-coastal-area",
    },
    (EXTRA_BIOSPHERE_DB, "sea-floor-transformation-oceanic-area"): {
        "name": "Sea floor transformation, oceanic area",
        "unit": "square meter",
        "categories": ("natural resource", "land"),
        "type": "natural resource",
        "code": "sea-floor-transformation-oceanic-area",
    },
}

bd.Database(EXTRA_BIOSPHERE_DB).write(extra_biosphere_data)

print("Created biosphere database:", EXTRA_BIOSPHERE_DB)
print("Database metadata:")
print(bd.databases[EXTRA_BIOSPHERE_DB])


100%|██████████████████████████████████████████████| 2/2 [00:00<00:00, 11110.74it/s]

22:14:38+0200 [info     ] Vacuuming database            


Created biosphere database: extra-biosphere
Database metadata:
{'depends': [], 'backend': 'sqlite', 'number': 2, 'modified': '2026-05-28T22:14:38.814211', 'geocollections': [], 'searchable': True, 'processed': '2026-05-28T22:14:40.345141', 'dirty': False}


Inspect the two flow nodes. Notice that Brightway has added the `database` and internal `id` fields after writing the database.


In [3]:
for flow in bd.Database(EXTRA_BIOSPHERE_DB):
    print("-" * 72)
    print("key:", flow.key)
    print("id:", flow.id)
    print("name:", flow["name"])
    print("unit:", flow["unit"])
    print("type:", flow["type"])
    print("categories:", flow["categories"])


------------------------------------------------------------------------
key: ('extra-biosphere', 'sea-floor-transformation-coastal-area')
id: 318482222045016064
name: Sea floor transformation, coastal area
unit: square meter
type: natural resource
categories: ('natural resource', 'land')
------------------------------------------------------------------------
key: ('extra-biosphere', 'sea-floor-transformation-oceanic-area')
id: 318482222045016065
name: Sea floor transformation, oceanic area
unit: square meter
type: natural resource
categories: ('natural resource', 'land')


## 4. Register a custom LCIA method

The method characterizes only the two new flows. Existing methods will not know what to do with these flows unless they are given characterization factors.

For this toy marine midpoint:
- `Sea floor transformation, coastal area`: 10 points per square meter
- `Sea floor transformation, oceanic area`: 2 points per square meter


In [4]:
characterization_factors = [
    ((EXTRA_BIOSPHERE_DB, "sea-floor-transformation-coastal-area"), 10.0),
    ((EXTRA_BIOSPHERE_DB, "sea-floor-transformation-oceanic-area"), 2.0),
]

method = bd.Method(CUSTOM_METHOD)
method.register(
    unit="marine transformation points",
    description="Toy method for two custom sea-floor transformation flows.",
)
method.write(characterization_factors)

print("Registered method:", CUSTOM_METHOD)
print("Method metadata:", bd.methods[CUSTOM_METHOD])
print("Method CFs:")
for flow_key, cf in characterization_factors:
    flow = bd.get_node(database=flow_key[0], code=flow_key[1])
    print(f"  {flow['name']}: {cf} {bd.methods[CUSTOM_METHOD]['unit']} per {flow['unit']}")


Registered method: ('additional LCIA method', 'marine seafloor transformation midpoint')
Method metadata: {'unit': 'marine transformation points', 'description': 'Toy method for two custom sea-floor transformation flows.', 'abbreviation': 'additional-lcia-methodm.b59d7b1b37693cac2b0b49d8b5ecb272', 'num_cfs': 2, 'geocollections': ['world']}
Method CFs:
  Sea floor transformation, coastal area: 10.0 marine transformation points per square meter
  Sea floor transformation, oceanic area: 2.0 marine transformation points per square meter


## 5. Create a fake process database

The fake process produces one meter of an offshore cable installation. Per meter installed, it has two biosphere exchanges:
- transforms 2 square meters of coastal sea floor
- transforms 5 square meters of oceanic sea floor

There are no technosphere inputs in this example. That keeps the score easy to verify manually.


In [5]:
process_data = {
    ("some-db", "cable-installation"): {
        "name": "Training offshore cable installation",
        "reference product": "installed offshore cable section",
        "unit": "meter",
        "location": "GLO",
        "type": "process",
        "code": "cable-installation",
        "exchanges": [
            # Every process needs a production exchange for its reference product.
            {
                "input": ("some-db", "cable-installation"),
                "amount": 1.0,
                "type": "production",
                "name": "Training offshore cable installation",
                "product": "installed offshore cable section",
                "unit": "meter",
            },
            # Biosphere exchanges point to the custom flows created above.
            {
                "input": (EXTRA_BIOSPHERE_DB, "sea-floor-transformation-coastal-area"),
                "amount": 2.0,
                "type": "biosphere",
                "name": "Sea floor transformation, coastal area",
                "unit": "square meter",
            },
            {
                "input": (EXTRA_BIOSPHERE_DB, "sea-floor-transformation-oceanic-area"),
                "amount": 5.0,
                "type": "biosphere",
                "name": "Sea floor transformation, oceanic area",
                "unit": "square meter",
            },
        ],
    }
}

bd.Database("some-db").write(process_data)

print("Created process database:", "some-db")
print("Database metadata:")
print(bd.databases["some-db"])


100%|███████████████████████████████████████████████| 1/1 [00:00<00:00, 1144.42it/s]

22:14:40+0200 [info     ] Vacuuming database            


Created process database: some-db
Database metadata:
{'depends': ['extra-biosphere'], 'backend': 'sqlite', 'number': 1, 'modified': '2026-05-28T22:14:40.373256', 'geocollections': ['world'], 'searchable': True, 'processed': '2026-05-28T22:14:41.822856', 'dirty': False}


In [15]:
bd.databases

Databases dictionary with 3 object(s):
	bafu
	ecoinvent-3.10-biosphere
	h2_electrolysis

Brightway records the dependency from the process database to the extra biosphere database because the process has exchanges to those flow keys.


In [9]:
activity = bd.get_activity(("some-db", "cable-installation"))

print("Activity key:", activity.key)
print("Process database depends on:", bd.databases[PROCESS_DB].get("depends"))
print()

for exc in activity.exchanges():
    input_node = exc.input
    label = input_node.get("name", input_node.key)
    unit = input_node.get("unit", "")
    print(f"{exc['type']:10s} amount={exc['amount']:>6g} {unit:8s} input={label}")


Activity key: ('some-db', 'cable-installation')
Process database depends on: ['extra-biosphere']

production amount=     1 meter    input=Training offshore cable installation
biosphere  amount=     2 square meter input=Sea floor transformation, coastal area
biosphere  amount=     5 square meter input=Sea floor transformation, oceanic area


## 6. Run LCI and LCIA

The functional unit is one meter of the fake offshore cable installation.


In [10]:
functional_unit = {activity: 1.0}

lca = bc.LCA(functional_unit, CUSTOM_METHOD)
lca.lci()
lca.lcia()

print(f"LCIA score: {lca.score:.6g} {bd.methods[CUSTOM_METHOD]['unit']}")


LCIA score: 30 marine transformation points
